In physics, "qudo" typically refers to Quadratic Unconstrained Discrete Optimization (QUDO). This is a mathematical framework—often studied in quantum computing and physics-inspired computing—used to solve complex combinatorial problems by assigning variables to finite integer values (states) rather than just binary 0 or 1.

The Optimization Problem: In operations and mathematical physics, finding the lowest energy state of a system often equates to solving complex minimization problems. QUDO is the discrete, generalized version of QUBO (Quadratic Unconstrained Binary Optimization).

Portfolio Optimization: Models the trade-off between maximizing expected returns and minimizing variance/risk, mapping classical Mean-Variance problems into binary matrices for high-speed processing.

In [1]:
import itertools

import numpy as np

Q = np.array([[0, -1, -2], [0, -3, 3], [0, 0, 2]])

# Solve this small QUBO exactly by enumerating all binary states.
def brute_force_qubo(matrix):
    best_state = None
    best_value = float('inf')

    for bits in itertools.product([0, 1], repeat=matrix.shape[0]):
        state = np.array(bits)
        value = state @ matrix @ state
        if value < best_value:
            best_value = value
            best_state = state

    return best_state, best_value

best_state, best_value = brute_force_qubo(Q)
best_state, best_value

(array([1, 1, 0]), np.int64(-4))

What QUBO Is? QUBO (Quadratic Unconstrained Binary Optimization)
A QUBO problem minimizes an objective of the form:

$f(x) = x^T Q x = \sum_i Q_{ii} x_i + \sum_{i<j} Q_{ij} x_i x_j$

where $x_i \in \{0,1\}$

Q is a real symmetric (or upper-triangular) matrix. "Unconstrained" means there are no explicit constraints — any constraints (e.g., "exactly one asset selected") get folded into QQ
Q as penalty terms.
Why it matters computationally: QUBO is NP-hard in general, but it's the native problem format for:

Quantum annealers (D-Wave) — QUBO maps directly onto the hardware's qubit interaction graph.
Ising machines / simulated annealing — QUBO is trivially convertible to an Ising Hamiltonian via $x_i = (1+s_i)/2$
xi​=(1+si​)/2, turning it into $H(s) = \sum J_{ij} s_i s_j + \sum h_i s_i$
with spins $s_i \in \{-1,+1\}$.

This is the direct link to your spin-glass modeling — QUBO and Ising are the same optimization landscape in two variable encodings.

QUBO in Financial Markets
The mapping to binary decisions makes QUBO natural for combinatorial financial optimization:

Portfolio selection under cardinality constraints: choosing exactly kk
k assets from nn
n (binary inclusion variables) to optimize a Markowitz-style mean-variance tradeoff, with the cardinality constraint penalized into Q:


$f(x) = \lambda \, x^T \Sigma x - \mu^T x + P\left(\sum_i x_i - k\right)^2$
This is a direct extension of your Markowitz/efficient-frontier notebooks — same
Σ (covariance) and μ (returns), but with the discrete "which assets" decision instead of continuous weights.
Index tracking / basket replication: select a sparse subset of assets that tracks an index — a combinatorial subset-selection problem, natural for QUBO.
Arbitrage detection: some formulations of triangular/statistical arbitrage across many pairs reduce to combinatorial cycle-selection problems, encodable as QUBO.
Trading signal combination: selecting a subset of signals/features (from your ARIMA/GARCH/HMM/entropy stack) that jointly maximize predictive power while penalizing redundancy — a QUBO feature-selection formulation, analogous to how it's used in ML feature selection generally.


Risk parity / diversification under constraints: binary sector/asset-class allocation constraints can be folded in as quadratic penalty terms.

A. Cardinality-constrained Markowitz (asset selection)Binary $x_i \in \{0,1\}$ = "include asset
i." Objective:


$f(x) = \lambda\, x^T \Sigma x - \mu^T x + P_1\left(\sum_i x_i - k\right)^2$

Σ: covariance matrix,
μ: expected returns, 
λ: risk aversion,
k: target number of holdings.

$P_1$​: penalty coefficient enforcing cardinality — must be large enough that violating it always costs more than any gain in the objective.

B. Discretized weight allocation (more useful for actual portfolios)Continuous weights don't fit QUBO natively, so you binary-encode them. For each asset
i, use
m bits to represent a discretized weight:


$w_i = \sum_{b=0}^{m-1} 2^b x_{i,b} \cdot \delta$
where
δ is your resolution (e.g., 1% increments). Substitute into the standard mean-variance objective:


$f(x) = \lambda\ w(x)^T \Sigma\ w(x) - \mu^T w(x) + P_2\left(\sum_i w_i(x) - 1\right)^2$
The budget constraint $(\sum w_i = 1)$ becomes a penalty term, same as cardinality. This is the version that actually generalizes Markowitz rather than just doing subset selection.

Penalty Coefficient Tuning
This is the part people get wrong most often. P_1, P_2 need to be:

Large enough that no constraint violation ever beats a feasible solution
Not so large that they wash out the actual objective landscape and make the solver indifferent between good and bad feasible solutions

Practical approach: start with $P \approx (max eigenvalue of \Sigma) × (a small multiplier like 5–10)$, then sweep and check feasibility rate empirically.

Build the QUBO Matrix Programmatically
Don't hand-derive the
Q matrix — use pyqubo or dimod's BinaryQuadraticModel to define the objective and constraints symbolically, then compile to Q.
This avoids sign/indexing errors when your bit-encoding gets large (n_assets × m_bits per asset grows fast).

In [ ]:
import numpy as np
from pyqubo import Array, Constraint, Placeholder

num_assets = 3
bits_per_asset = 2
delta = 0.25

mu = np.array([0.10, 0.12, 0.08])
Sigma = np.array([
    [0.10, 0.02, 0.04],
    [0.02, 0.08, 0.01],
    [0.04, 0.01, 0.09],
])

n, m = num_assets, bits_per_asset
x = Array.create('x', shape=(n, m), vartype='BINARY')
w = [sum(2**b * x[i][b] for b in range(m)) * delta for i in range(n)]

risk = sum(Sigma[i][j] * w[i] * w[j] for i in range(n) for j in range(n))
ret = sum(mu[i] * w[i] for i in range(n))
budget_penalty = Constraint((sum(w) - 1)**2, label='budget')

H = Placeholder('lambda') * risk - ret + Placeholder('P') * budget_penalty
model = H.compile()
Q, offset = model.to_qubo(feed_dict={'lambda': 0.5, 'P': 50})
Q, offset

c:\Users\saiga\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyqubo\integer\order_enc_integer.py:21: SyntaxWarning: invalid escape sequence '\s'
  """
c:\Users\saiga\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyqubo\integer\one_hot_enc_integer.py:21: SyntaxWarning: invalid escape sequence '\s'
  """One-hot encoded integer. The value that takes :math:`[1, n]` is represented by :math:`\sum_{i=1}^{n}ix_{i}`.
c:\Users\saiga\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyqubo\integer\unary_encoded_integer.py:21: SyntaxWarning: invalid escape sequence '\s'
  """Unary encoded integer. The value that takes :math:`[0, n]` is


({('x[2][1]', 'x[1][1]'): 25.0025,
  ('x[0][1]', 'x[0][1]'): -37.537499999999994,
  ('x[2][0]', 'x[1][1]'): 12.50125,
  ('x[2][1]', 'x[0][0]'): 12.505,
  ('x[2][0]', 'x[1][0]'): 6.250625,
  ('x[2][1]', 'x[1][0]'): 12.50125,
  ('x[1][0]', 'x[0][1]'): 12.5025,
  ('x[1][0]', 'x[0][0]'): 6.25125,
  ('x[2][0]', 'x[0][0]'): 6.2525,
  ('x[0][1]', 'x[0][0]'): 12.5125,
  ('x[1][1]', 'x[0][0]'): 12.5025,
  ('x[2][1]', 'x[2][0]'): 12.51125,
  ('x[1][1]', 'x[1][0]'): 12.51,
  ('x[2][0]', 'x[2][0]'): -21.8921875,
  ('x[1][1]', 'x[0][1]'): 25.005,
  ('x[1][1]', 'x[1][1]'): -37.550000000000004,
  ('x[2][0]', 'x[0][1]'): 12.505,
  ('x[2][1]', 'x[2][1]'): -37.52875,
  ('x[0][0]', 'x[0][0]'): -21.896874999999998,
  ('x[2][1]', 'x[0][1]'): 25.01,
  ('x[1][0]', 'x[1][0]'): -21.9025},
 50.0)

: 